# OPRB Comparative Benchmark Analysis & Report Generator

This notebook aggregates the evaluation results from all **05 lightweight models**:
1. **PP-OCR Mobile** (~15M, OCR Baseline)
2. **EasyOCR** (~100M, OCR Baseline)
3. **Granite-Docling-258M** (258M, Document VLM)
4. **PaddleOCR-VL-1.6** (0.9B, Document VLM)
5. **Qwen2.5-VL-3B-Instruct** (3B, General VLM)

It generates:
- **Master Quantitative Table** (CER, WER, Precision, Recall, F1, Latency, Model Size)
- **Performance Comparison**: `text-region` vs `nontext-region`
- **Robustness Analysis**: Degradation by occlusion type & density
- **Efficiency Trade-off**: Pareto chart (Model size & Latency vs WER)
- **Qualitative Case Studies**: Recovery vs Hallucination examples
- **LaTeX Code**: Publication-ready table to paste directly into your 4-page report!

In [ ]:
import json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML

# Configure styling for academic paper quality
plt.rcParams.update({
    'font.size': 12,
    'axes.labelsize': 13,
    'axes.titlesize': 14,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'figure.titlesize': 15,
    'figure.dpi': 200,
})

OUTPUT_DIR = Path("report_assets")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Asset directory created:", OUTPUT_DIR)

In [ ]:
# Define model configurations and CSV lookup paths
MODEL_CONFIGS = [
    {
        "name": "PP-OCR Mobile",
        "type": "OCR (2-stage)",
        "params_m": 15,
        "csv_path": Path("/content/oprb_paddle_results/per_image_results.csv"),
        "local_csv": Path("oprb_paddle_results/per_image_results.csv"),
    },
    {
        "name": "EasyOCR",
        "type": "OCR (2-stage)",
        "params_m": 100,
        "csv_path": Path("/content/oprb_easyocr_results/per_image_results.csv"),
        "local_csv": Path("oprb_easyocr_results/per_image_results.csv"),
    },
    {
        "name": "Granite-Docling-258M",
        "type": "Doc-VLM",
        "params_m": 258,
        "csv_path": Path("/content/oprb_docling_results/per_image_results.csv"),
        "local_csv": Path("oprb_docling_results/per_image_results.csv"),
    },
    {
        "name": "PaddleOCR-VL-1.6",
        "type": "Doc-VLM",
        "params_m": 900,
        "csv_path": Path("/content/oprb_paddlevl_results/per_image_results.csv"),
        "local_csv": Path("oprb_paddlevl_results/per_image_results.csv"),
    },
    {
        "name": "Qwen2.5-VL-3B",
        "type": "General VLM",
        "params_m": 3000,
        "csv_path": Path("/content/oprb_qwen_results/per_image_results.csv"),
        "local_csv": Path("oprb_qwen_results/per_image_results.csv"),
    },
]

dfs = []
for m in MODEL_CONFIGS:
    p = m["csv_path"] if m["csv_path"].exists() else m["local_csv"]
    if p.exists():
        df = pd.read_csv(p)
        df["model_name"] = m["name"]
        df["model_type"] = m["type"]
        df["params_m"] = m["params_m"]
        dfs.append(df)
        print(f"Loaded {m['name']}: {len(df)} samples from {p}")
    else:
        print(f"[Pending run] {m['name']} results not found yet at {p}")

if dfs:
    all_results = pd.concat(dfs, ignore_index=True)
    all_ok = all_results[all_results["status"] == "ok"].copy()
    print(f"Total valid evaluation records: {len(all_ok):,}")
else:
    print("No CSV files loaded yet. Run the evaluation notebooks first!")

## 1. Master Quantitative Results Table

In [ ]:
metric_cols = ['cer', 'wer', 'token_precision', 'token_recall', 'token_f1', 'latency_sec']

if 'all_ok' in locals() and not all_ok.empty:
    master_table = all_ok.groupby(['model_name', 'model_type', 'params_m']).agg(
        evaluated_pages=('sample_id', 'count'),
        cer=('cer', 'mean'),
        wer=('wer', 'mean'),
        precision=('token_precision', 'mean'),
        recall=('token_recall', 'mean'),
        f1=('token_f1', 'mean'),
        latency=('latency_sec', 'mean'),
    ).reset_index().sort_values('wer')
    
    display(master_table.round(4))
    master_table.to_csv(OUTPUT_DIR / "master_summary_table.csv", index=False)
    
    # Generate LaTeX table code for paper submission
    latex_code = master_table.to_latex(index=False, float_format="%.4f")
    (OUTPUT_DIR / "master_table.tex").write_text(latex_code)
    print("Saved LaTeX table to:", OUTPUT_DIR / "master_table.tex")

## 2. Text-region vs Non-text-region Occlusions

In [ ]:
if 'all_ok' in locals() and not all_ok.empty:
    region_summary = all_ok.groupby(['model_name', 'category']).agg(
        cer=('cer', 'mean'),
        wer=('wer', 'mean'),
        f1=('token_f1', 'mean'),
    ).reset_index()
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # WER Plot
    sns.barplot(data=region_summary, x='model_name', y='wer', hue='category', ax=axes[0], palette='Blues_r')
    axes[0].set_title('Word Error Rate (WER) by Occlusion Region')
    axes[0].set_ylabel('WER (lower is better)')
    axes[0].set_xlabel('')
    axes[0].tick_params(axis='x', rotation=15)
    
    # Token F1 Plot
    sns.barplot(data=region_summary, x='model_name', y='f1', hue='category', ax=axes[1], palette='Greens_r')
    axes[1].set_title('Token F1-Score by Occlusion Region')
    axes[1].set_ylabel('F1-Score (higher is better)')
    axes[1].set_xlabel('')
    axes[1].tick_params(axis='x', rotation=15)
    
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "fig1_region_comparison.png", dpi=200)
    plt.show()

## 3. Robustness Analysis across Occlusion Types

In [ ]:
if 'all_ok' in locals() and not all_ok.empty:
    type_summary = all_ok.groupby(['occlusion_type', 'model_name'])['wer'].mean().unstack()
    
    plt.figure(figsize=(12, 6))
    type_summary.plot(kind='bar', ax=plt.gca(), width=0.8)
    plt.title('Robustness by Occlusion Type (Mean WER)')
    plt.ylabel('Mean WER (lower is better)')
    plt.xlabel('Occlusion Type')
    plt.xticks(rotation=20, ha='right')
    plt.legend(title='Model')
    plt.grid(axis='y', linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "fig2_robustness_by_type.png", dpi=200)
    plt.show()

## 4. Efficiency Trade-offs: Model Size & Latency vs Accuracy

In [ ]:
if 'all_ok' in locals() and not all_ok.empty:
    plt.figure(figsize=(9, 5.5))
    
    for idx, row in master_table.iterrows():
        plt.scatter(
            row['latency'], 
            row['wer'], 
            s=row['params_m'] * 0.8 + 100,
            alpha=0.65,
            edgecolors='black',
            linewidth=1.5,
            label=f"{row['model_name']} ({row['params_m']}M)"
        )
        plt.text(row['latency'] * 1.05, row['wer'], f" {row['model_name']}", fontsize=11, va='center')
    
    plt.xscale('log')
    plt.title('Pareto Frontier: Inference Latency vs Word Error Rate (WER)')
    plt.xlabel('Average Latency per Page (seconds, log scale)')
    plt.ylabel('Mean WER (lower is better)')
    plt.grid(True, which="both", ls="--", alpha=0.5)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "fig3_pareto_efficiency.png", dpi=200)
    plt.show()

## 5. Qualitative Case Studies (Recovery vs Hallucination)

In [ ]:
if 'all_ok' in locals() and not all_ok.empty:
    # Identify cases where VLM drastically outperformed OCR (Recovery Case)
    pivot_wer = all_ok.pivot_table(index='sample_id', columns='model_name', values='wer')
    if 'Qwen2.5-VL-3B' in pivot_wer and 'EasyOCR' in pivot_wer:
        pivot_wer['vlm_advantage'] = pivot_wer['EasyOCR'] - pivot_wer['Qwen2.5-VL-3B']
        best_recovery_samples = pivot_wer.sort_values('vlm_advantage', ascending=False).head(3).index.tolist()
        print("Top samples where VLM recovered occluded text better than OCR:", best_recovery_samples)
        
        for sid in best_recovery_samples:
            sub = all_ok[all_ok['sample_id'] == sid]
            gt = sub['ground_truth'].iloc[0]
            print(f"\n--- Sample ID: {sid} (Occlusion: {sub['occlusion_type'].iloc[0]}) ---")
            print("Ground Truth (first 250 chars):", gt[:250])
            for _, r in sub.iterrows():
                print(f"[{r['model_name']}] (WER: {r['wer']:.3f}, CER: {r['cer']:.3f}): {r['prediction'][:250]}")